In [21]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("PySpark Practice") \
    .master("local[*]") \
    .getOrCreate()

print("Spark Session Started")

Spark Session Started


In [22]:
# Importing necessary functions from PySpark SQL module
from pyspark.sql.functions import col
from pyspark.sql.functions import when
from pyspark.sql.functions import upper
from pyspark.sql.functions import lower
from pyspark.sql.functions import year
from pyspark.sql.functions import current_date
from pyspark.sql.functions import split


In [23]:
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("customers.csv")

In [24]:
df.show(5)

+-----------+-------------+--------------------+------------+---+--------------+---------------+-----------+---------+
|customer_id|    full_name|               email|        city|age|lifetime_value|membership_tier|signup_date|is_active|
+-----------+-------------+--------------------+------------+---+--------------+---------------+-----------+---------+
|          1|  Sarah Smith|sarah.smith1@exam...|   Waterbury| 33|       1154.89|         Bronze| 2026-02-27|     true|
|          2|Melissa Smith|melissa.smith2@ex...|East Windsor| 31|       1201.67|         Bronze| 2022-03-03|     true|
|          3|Melissa Brown|melissa.brown3@ex...|   Waterbury| 69|       4353.04|         Silver| 2024-07-08|    false|
|          4|Matthew Jones|matthew.jones4@ex...|    Stamford| 66|       1716.14|         Bronze| 2022-11-15|    false|
|          5| Sarah Garcia|sarah.garcia5@exa...|   Waterbury| 69|        265.08|       Platinum| 2023-12-06|     true|
+-----------+-------------+--------------------+

In [25]:
# Filtering the DataFrame to get only active customers
df_active = df.filter(col("is_active") == True)
df_active.show()

+-----------+--------------+--------------------+------------+---+--------------+---------------+-----------+---------+
|customer_id|     full_name|               email|        city|age|lifetime_value|membership_tier|signup_date|is_active|
+-----------+--------------+--------------------+------------+---+--------------+---------------+-----------+---------+
|          1|   Sarah Smith|sarah.smith1@exam...|   Waterbury| 33|       1154.89|         Bronze| 2026-02-27|     true|
|          2| Melissa Smith|melissa.smith2@ex...|East Windsor| 31|       1201.67|         Bronze| 2022-03-03|     true|
|          5|  Sarah Garcia|sarah.garcia5@exa...|   Waterbury| 69|        265.08|       Platinum| 2023-12-06|     true|
|          6| James Johnson|james.johnson6@ex...|   Waterbury| 71|       3161.77|           Gold| 2025-02-04|     true|
|          7| Michael Smith|michael.smith7@ex...|    Stamford| 67|       1482.47|         Bronze| 2025-09-16|     true|
|          9|  Amanda Brown|amanda.brown

In [26]:
df_over_70 = df.filter(col("age") > 70)

df_over_70.show()

+-----------+----------------+--------------------+------------+---+--------------+---------------+-----------+---------+
|customer_id|       full_name|               email|        city|age|lifetime_value|membership_tier|signup_date|is_active|
+-----------+----------------+--------------------+------------+---+--------------+---------------+-----------+---------+
|          6|   James Johnson|james.johnson6@ex...|   Waterbury| 71|       3161.77|           Gold| 2025-02-04|     true|
|         31|   Melissa Davis|melissa.davis31@e...|    Stamford| 73|       2035.15|         Bronze| 2024-09-12|     true|
|         41|  Sarah Williams|sarah.williams41@...|East Windsor| 74|        579.83|         Silver| 2023-06-26|    false|
|         43|    Daniel Smith|daniel.smith43@ex...|     Bristol| 71|       1419.57|         Bronze| 2022-07-09|    false|
|         45|Michael Williams|michael.williams4...|    Hartford| 71|       1877.62|         Silver| 2025-01-22|    false|
|         46|     David 

In [27]:
df_names = df.select(
    "customer_id",
    "full_name",
    "city"
)

df_names.show(3)

+-----------+-------------+------------+
|customer_id|    full_name|        city|
+-----------+-------------+------------+
|          1|  Sarah Smith|   Waterbury|
|          2|Melissa Smith|East Windsor|
|          3|Melissa Brown|   Waterbury|
+-----------+-------------+------------+
only showing top 3 rows


In [28]:
df_bonus = df.withColumn(
    "new_lifetime_value",
    col("lifetime_value") * 1.10
)

df_bonus.show(3)


+-----------+-------------+--------------------+------------+---+--------------+---------------+-----------+---------+------------------+
|customer_id|    full_name|               email|        city|age|lifetime_value|membership_tier|signup_date|is_active|new_lifetime_value|
+-----------+-------------+--------------------+------------+---+--------------+---------------+-----------+---------+------------------+
|          1|  Sarah Smith|sarah.smith1@exam...|   Waterbury| 33|       1154.89|         Bronze| 2026-02-27|     true|1270.3790000000001|
|          2|Melissa Smith|melissa.smith2@ex...|East Windsor| 31|       1201.67|         Bronze| 2022-03-03|     true|1321.8370000000002|
|          3|Melissa Brown|melissa.brown3@ex...|   Waterbury| 69|       4353.04|         Silver| 2024-07-08|    false|          4788.344|
+-----------+-------------+--------------------+------------+---+--------------+---------------+-----------+---------+------------------+
only showing top 3 rows


In [29]:
df_segment = df.withColumn(
    "customer_segment",
    when(col("lifetime_value") > 5000, "High Value")
    .otherwise("Standard")
)

df_segment.show(3)

+-----------+-------------+--------------------+------------+---+--------------+---------------+-----------+---------+----------------+
|customer_id|    full_name|               email|        city|age|lifetime_value|membership_tier|signup_date|is_active|customer_segment|
+-----------+-------------+--------------------+------------+---+--------------+---------------+-----------+---------+----------------+
|          1|  Sarah Smith|sarah.smith1@exam...|   Waterbury| 33|       1154.89|         Bronze| 2026-02-27|     true|        Standard|
|          2|Melissa Smith|melissa.smith2@ex...|East Windsor| 31|       1201.67|         Bronze| 2022-03-03|     true|        Standard|
|          3|Melissa Brown|melissa.brown3@ex...|   Waterbury| 69|       4353.04|         Silver| 2024-07-08|    false|        Standard|
+-----------+-------------+--------------------+------------+---+--------------+---------------+-----------+---------+----------------+
only showing top 3 rows


In [30]:
# conditional column creation based on lifetime_value
df_segment = df \
    .withColumn(
        "customer_segment",
        when(col("lifetime_value") > 5000, "High Value")
        .otherwise("Standard")
    )

df_segment.show()

+-----------+----------------+--------------------+------------+---+--------------+---------------+-----------+---------+----------------+
|customer_id|       full_name|               email|        city|age|lifetime_value|membership_tier|signup_date|is_active|customer_segment|
+-----------+----------------+--------------------+------------+---+--------------+---------------+-----------+---------+----------------+
|          1|     Sarah Smith|sarah.smith1@exam...|   Waterbury| 33|       1154.89|         Bronze| 2026-02-27|     true|        Standard|
|          2|   Melissa Smith|melissa.smith2@ex...|East Windsor| 31|       1201.67|         Bronze| 2022-03-03|     true|        Standard|
|          3|   Melissa Brown|melissa.brown3@ex...|   Waterbury| 69|       4353.04|         Silver| 2024-07-08|    false|        Standard|
|          4|   Matthew Jones|matthew.jones4@ex...|    Stamford| 66|       1716.14|         Bronze| 2022-11-15|    false|        Standard|
|          5|    Sarah Garc

In [31]:
df_drop = df \
    .drop("email")

df_drop.show()

+-----------+----------------+------------+---+--------------+---------------+-----------+---------+
|customer_id|       full_name|        city|age|lifetime_value|membership_tier|signup_date|is_active|
+-----------+----------------+------------+---+--------------+---------------+-----------+---------+
|          1|     Sarah Smith|   Waterbury| 33|       1154.89|         Bronze| 2026-02-27|     true|
|          2|   Melissa Smith|East Windsor| 31|       1201.67|         Bronze| 2022-03-03|     true|
|          3|   Melissa Brown|   Waterbury| 69|       4353.04|         Silver| 2024-07-08|    false|
|          4|   Matthew Jones|    Stamford| 66|       1716.14|         Bronze| 2022-11-15|    false|
|          5|    Sarah Garcia|   Waterbury| 69|        265.08|       Platinum| 2023-12-06|     true|
|          6|   James Johnson|   Waterbury| 71|       3161.77|           Gold| 2025-02-04|     true|
|          7|   Michael Smith|    Stamford| 67|       1482.47|         Bronze| 2025-09-16| 

In [32]:
df_upper = df \
    .withColumn(
        "city",
        upper(col("city"))
    )

df_upper.show()

+-----------+----------------+--------------------+------------+---+--------------+---------------+-----------+---------+
|customer_id|       full_name|               email|        city|age|lifetime_value|membership_tier|signup_date|is_active|
+-----------+----------------+--------------------+------------+---+--------------+---------------+-----------+---------+
|          1|     Sarah Smith|sarah.smith1@exam...|   WATERBURY| 33|       1154.89|         Bronze| 2026-02-27|     true|
|          2|   Melissa Smith|melissa.smith2@ex...|EAST WINDSOR| 31|       1201.67|         Bronze| 2022-03-03|     true|
|          3|   Melissa Brown|melissa.brown3@ex...|   WATERBURY| 69|       4353.04|         Silver| 2024-07-08|    false|
|          4|   Matthew Jones|matthew.jones4@ex...|    STAMFORD| 66|       1716.14|         Bronze| 2022-11-15|    false|
|          5|    Sarah Garcia|sarah.garcia5@exa...|   WATERBURY| 69|        265.08|       Platinum| 2023-12-06|     true|
|          6|   James Jo

In [33]:
df_lower = df \
    .withColumn(
        "membership_tier",
        lower(col("membership_tier"))
    )

df_lower.show()

+-----------+----------------+--------------------+------------+---+--------------+---------------+-----------+---------+
|customer_id|       full_name|               email|        city|age|lifetime_value|membership_tier|signup_date|is_active|
+-----------+----------------+--------------------+------------+---+--------------+---------------+-----------+---------+
|          1|     Sarah Smith|sarah.smith1@exam...|   Waterbury| 33|       1154.89|         bronze| 2026-02-27|     true|
|          2|   Melissa Smith|melissa.smith2@ex...|East Windsor| 31|       1201.67|         bronze| 2022-03-03|     true|
|          3|   Melissa Brown|melissa.brown3@ex...|   Waterbury| 69|       4353.04|         silver| 2024-07-08|    false|
|          4|   Matthew Jones|matthew.jones4@ex...|    Stamford| 66|       1716.14|         bronze| 2022-11-15|    false|
|          5|    Sarah Garcia|sarah.garcia5@exa...|   Waterbury| 69|        265.08|       platinum| 2023-12-06|     true|
|          6|   James Jo

In [34]:
df_sorted = df \
    .orderBy(
        col("lifetime_value").desc()
    )

df_sorted.show()

+-----------+----------------+--------------------+------------+---+--------------+---------------+-----------+---------+
|customer_id|       full_name|               email|        city|age|lifetime_value|membership_tier|signup_date|is_active|
+-----------+----------------+--------------------+------------+---+--------------+---------------+-----------+---------+
|         38| Matthew Johnson|matthew.johnson38...|     Danbury| 57|       4992.35|         Bronze| 2022-01-20|     true|
|         90|   Nicole Wilson|nicole.wilson90@e...|     Norwalk| 45|       4983.33|           Gold| 2024-12-23|    false|
|         97|   Jane Williams|jane.williams97@e...|   Waterbury| 23|       4969.89|         Bronze| 2022-11-02|    false|
|         10|Jessica Williams|jessica.williams1...|     Bristol| 35|       4948.14|         Silver| 2024-08-04|    false|
|         40| Ashley Williams|ashley.williams40...|   Waterbury| 57|       4920.07|         Bronze| 2024-06-16|    false|
|         33| Nicole Wil

In [35]:
df_city_avg = df \
    .groupBy("city") \
    .avg("lifetime_value")

df_city_avg.show()

+------------+-------------------+
|        city|avg(lifetime_value)|
+------------+-------------------+
|East Windsor| 2400.8500000000004|
|    Stamford| 2318.5863636363633|
|     Bristol| 2517.5633333333335|
|     Danbury|           3197.523|
|   Waterbury| 2603.4255555555555|
|   New Haven| 1856.2354545454546|
|    Hartford| 2322.7607142857146|
|     Norwalk|  3670.791818181818|
+------------+-------------------+



In [36]:
df_tier_counts = df \
    .groupBy("membership_tier") \
    .count()

df_tier_counts.show()

+---------------+-----+
|membership_tier|count|
+---------------+-----+
|       Platinum|   24|
|         Silver|   22|
|           Gold|   29|
|         Bronze|   25|
+---------------+-----+



In [37]:
df_active_city = df \
    .filter(col("is_active") == True) \
    .groupBy("city") \
    .count()

df_active_city.show()

+------------+-----+
|        city|count|
+------------+-----+
|East Windsor|    8|
|    Stamford|    6|
|     Bristol|    4|
|     Danbury|    5|
|   Waterbury|    7|
|   New Haven|    6|
|    Hartford|    7|
|     Norwalk|    3|
+------------+-----+



In [38]:
df_year = df \
    .withColumn(
        "signup_year",
        year(col("signup_date"))
    )

df_year.show()

+-----------+----------------+--------------------+------------+---+--------------+---------------+-----------+---------+-----------+
|customer_id|       full_name|               email|        city|age|lifetime_value|membership_tier|signup_date|is_active|signup_year|
+-----------+----------------+--------------------+------------+---+--------------+---------------+-----------+---------+-----------+
|          1|     Sarah Smith|sarah.smith1@exam...|   Waterbury| 33|       1154.89|         Bronze| 2026-02-27|     true|       2026|
|          2|   Melissa Smith|melissa.smith2@ex...|East Windsor| 31|       1201.67|         Bronze| 2022-03-03|     true|       2022|
|          3|   Melissa Brown|melissa.brown3@ex...|   Waterbury| 69|       4353.04|         Silver| 2024-07-08|    false|       2024|
|          4|   Matthew Jones|matthew.jones4@ex...|    Stamford| 66|       1716.14|         Bronze| 2022-11-15|    false|       2022|
|          5|    Sarah Garcia|sarah.garcia5@exa...|   Waterbur

In [39]:
from pyspark.sql.functions import datediff
from pyspark.sql.functions import current_date

df_days = df \
    .withColumn(
        "days_as_customer",
        datediff(
            current_date(),
            col("signup_date")
        )
    )

df_days.show()

+-----------+----------------+--------------------+------------+---+--------------+---------------+-----------+---------+----------------+
|customer_id|       full_name|               email|        city|age|lifetime_value|membership_tier|signup_date|is_active|days_as_customer|
+-----------+----------------+--------------------+------------+---+--------------+---------------+-----------+---------+----------------+
|          1|     Sarah Smith|sarah.smith1@exam...|   Waterbury| 33|       1154.89|         Bronze| 2026-02-27|     true|             234|
|          2|   Melissa Smith|melissa.smith2@ex...|East Windsor| 31|       1201.67|         Bronze| 2022-03-03|     true|            1691|
|          3|   Melissa Brown|melissa.brown3@ex...|   Waterbury| 69|       4353.04|         Silver| 2024-07-08|    false|             833|
|          4|   Matthew Jones|matthew.jones4@ex...|    Stamford| 66|       1716.14|         Bronze| 2022-11-15|    false|            1434|
|          5|    Sarah Garc

In [40]:
df = df \
    .withColumn(
        "first_name",
        split(col("full_name"), " ").getItem(0)
    ) \
    .withColumn(
        "last_name",
        split(col("full_name"), " ").getItem(1)
    )

df.show(5)

+-----------+-------------+--------------------+------------+---+--------------+---------------+-----------+---------+----------+---------+
|customer_id|    full_name|               email|        city|age|lifetime_value|membership_tier|signup_date|is_active|first_name|last_name|
+-----------+-------------+--------------------+------------+---+--------------+---------------+-----------+---------+----------+---------+
|          1|  Sarah Smith|sarah.smith1@exam...|   Waterbury| 33|       1154.89|         Bronze| 2026-02-27|     true|     Sarah|    Smith|
|          2|Melissa Smith|melissa.smith2@ex...|East Windsor| 31|       1201.67|         Bronze| 2022-03-03|     true|   Melissa|    Smith|
|          3|Melissa Brown|melissa.brown3@ex...|   Waterbury| 69|       4353.04|         Silver| 2024-07-08|    false|   Melissa|    Brown|
|          4|Matthew Jones|matthew.jones4@ex...|    Stamford| 66|       1716.14|         Bronze| 2022-11-15|    false|   Matthew|    Jones|
|          5| Sarah 